## OpenAI API 활용 1

#### 1. 챗봇 만들기

In [ ]:
import streamlit as st
from openai import OpenAI
from dotenv import load_dotenv
import os
import datetime
import json

load_dotenv()

OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')

client = OpenAI(api_key=OPENAI_API_KEY)

##### 함수 정의

In [ ]:
def get_today_date():
    today = datetime.datetime.now().strftime('%Y-%m-%d')
    return today

##### tools 정의

In [ ]:
tools = [
    {
        "type": "function",
        "name": "get_today_date",
        "description": "오늘 날짜를 YYYY-MM-DD 형식의 문자열로 반환합니다.",
        "parameters": {
            "type": "object",
            "properties": {},
            "required": []
        }
    }
]

##### function_call 처리

In [ ]:
# st.session_state : 세션에 키-값 형식으로 데이터를 저장하는 변수
# openai_model: str, message: []
if 'openai_model' not in st.session_state:
    st.session_state.openai_model = 'gpt-4.1'   # 'gpt-3.5-turbo'

if 'messages' not in st.session_state:
    st.session_state.messages = []

# 기존의 메시지가 있다면 출력
for msg in st.session_state.messages:
    with st.chat_message(msg['role']):
        st.markdown(msg['content'])

# prompt: 사용자 입력창
if prompt:= st.chat_input('메시지를 입력하세요!') :
    # st.write(prompt)
    st.session_state.messages.append({
        "role": "user",
        "content": prompt
    })

    with st.chat_message('user'):
        st.markdown(prompt)

    # 첫 번째 openai 요청 -> 함수 선택 -> 결과를 이용해서 재 요청
    response = client.responses.create(
        model=st.session_state.openai_model,
        input=st.session_state.messages,
        tools=tools
    )

    # 함수 호출 처리
    msg_content = None      # 최종 응답 결과
    tool_executed = False   # 함수 호출 여부

    if response.output:
        for tool_call in response.output:
            print(f'Tool 호출: {tool_call}')
            # type == function_call and name='get_today_date'
            if tool_call.type == 'function_call' and tool_call.name == 'get_today_date':
                print(f'Function call: {tool_call.name}')
                args = json.loads(tool_call.arguments or "{}")
                result = get_today_date()
                tool_executed = True

                st.session_state.messages.append({
                    "type": "function_call",
                    "call_id": tool_call.call_id,
                    "name": tool_call.name,
                    "arguments": tool_call.arguments
                })

                st.session_state.messages.append(
                    {
                        "type": "function_call_output",
                        "call_id": tool_call.call_id,
                        "output": result
                    }
                )
    if tool_executed:
        response2 = client.responses.create(
            model=st.session_state.openai_model,
            input=st.session_state.messages,
            tools=tools
        )

        msg_content = getattr(response2, 'output_text', None)
        
        with st.chat_message('assistant'):
            st.markdown(msg_content)
        
        # messages -> append
        # st.session_state.messages.append({
        #     "role": "assistants",
        #     "content": msg_content
        # })

    else:
        msg_content = getattr(response, 'output_text', None)
        with st.chat_message('assistant'):
            st.markdown(msg_content)

    st.session_state.messages.append({
        "role": "assistant",
        "content": msg_content
    })

In [ ]:
if 'messages' not in st.session_state:
    st.session_state.messages = []

# 기존의 메시지가 있다면 출력
for msg in st.session_state.messages:
    if msg.get('role') in ('user', 'assistant'):
        with st.chat_message(msg['role']):
            st.markdown(msg['content'])

if prompt := st.chat_input('메시지를 입력하세요!'):
    st.session_state.messages.append({
        "role": "user",
        "content": prompt
    })

    with st.chat_message('user'):
        st.markdown(prompt)
    
    with st.chat_message('assistant'):
        stream = client.chat.completions.create(
            model=st.session_state.openai_model,
            messages=[
                {"role": m['role'], "content": m['content']}
                for m in st.session_state.messages
            ],
            stream=True
        )
        response = st.write_stream(stream)

    st.session_state.messages.append(
        {
            "role": "assistant",
            "content": response
        }
    )

#### Streamlit과 Keras모델을 활용하여 개/고양이 분류기 만들기
- 실행 : `streamlit run app.py`

In [ ]:
# app.py
import streamlit as st
import tensorflow as tf
from PIL import Image, UnidentifiedImageError
import numpy as np

# 모델 로드
def load_model():
    try:
        model = tf.keras.models.load_model('cat_dog_classifier.keras')
        st.success('모델을 load 했습니다.')
        return model
    except:
        st.error('모델을 로드할 수 없습니다. 경로를 확인해주세요!')

model = load_model()

# 사용자가 업로드한 이미지 전처리
def preprocess_image(image):
    try:
        image = image.resize((150, 150))
        image = np.array(image) / 255.0     # 정규화
        if image.shape[-1] != 3:
            raise ValueError('이미지는 RGB 형식의 컬러이미지만 처리가 가능합니다.')
        image = np.expand_dims(image, axis=0)   # (1, 150, 150, 3)
        
        return image
    
    except Exception as e:
        st.error('이미지 전처리 중 문제가 발생했습니다.: {e}')
        
        return None
# UI
st.title('Cat/Dog 분류기')
st.write('이미지를 업로드 하면 개 또는 고양이를 판별합니다.')

uploadfile = st.file_uploader('이미지를 업로드하세요!', type=['jpg', 'png', 'jpeg'])

if uploadfile:
    try:
        # 이미지 로드 -> 이미지 파일을 이미기 객체로 변환
        image = Image.open(uploadfile)
        st.image(image, caption='업로드된 이미지', use_column_width=True)

        preprocessed_image = preprocess_image(image)

        if preprocess_image:
            preditcion = model.predict(preprocessed_image)
            print(preditcion)

        # 결과 표시
        if preditcion[0][0] > 0.5:
            st.success('이 이미지는 개로 분류되었습니다.')
        else:
            st.success('이 이미지는 고양이로 분류되었습니다.')
    except UnidentifiedImageError:
        st.error('이미지를 로드할 수 없습니다. 지원되지 않는 파일 형식입니다.', icon="🚨")
    except Exception as e:
        st.error(f'에측 처리 중 오류 발생 : {e}', icon="🚨")
    
